# 03 — Beitragsextraktion (SHAP / EBM)

Extrahiert globale Feature-Importance und lokale Beiträge für die **5 fixen Testinstanzen**
(`INSTANCE_IDS = [42, 100, 250, 500, 1337]`) für XGBoost und EBM mit **Option 2 (Poisson-Log)**.

- **XGBoost**: SHAP TreeExplainer — Beiträge im Log-Raum (additiv)
- **EBM**: interpret `explain_local` — Terme ebenfalls im Log-Raum

Alle Ausgaben werden als JSON in `explanations/` gespeichert und von Notebooks 04–06 gelesen.

In [1]:
from __future__ import annotations

import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd

from utils import INSTANCE_IDS, EXPLANATIONS_DIR, MODELS_DIR, RESULTS_DIR
from utils.data import load_train_test
from utils.explanations import build_global, build_local, save_explanation

LOSS_KEY = "poisson_log"

print(f"Testinstanzen: {INSTANCE_IDS}")
print(f"Loss-Option:   {LOSS_KEY}")
print(f"Ausgabe:       {EXPLANATIONS_DIR}")

Testinstanzen: [224, 580, 1041, 1481, 1677, 2058, 2510, 3543, 3847, 4454]
Loss-Option:   poisson_log
Ausgabe:       /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/explanations


## 1 · Daten und Modelle laden

In [2]:
X_train, y_train, X_test, y_test = load_train_test()
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

from utils.models import load_models
xgb, ebm = load_models(LOSS_KEY)
print("Modelle geladen.")

metrics = json.loads((RESULTS_DIR / f"model_metrics_{LOSS_KEY}.json").read_text())["metrics"]
print("Metriken geladen:", list(metrics.keys()))

X_train: (12152, 9)  |  X_test: (5227, 9)
Modelle geladen.
Metriken geladen: ['xgb', 'ebm']


## 2 · Globale Erklärungen

In [3]:
print("Berechne globale XGBoost-Erklärung (SHAP über Trainingsset)...")
global_xgb = build_global(xgb, "xgb", X_train, metrics["xgb"])

path = save_explanation(global_xgb, f"global_xgb_{LOSS_KEY}.json")
print(f"Gespeichert: {path}")

print("\nTop-5 Features (XGBoost):")
for entry in global_xgb["global_importance"][:5]:
    print(f"  #{entry['rank']:2d}  {entry['feature']:12s}  {entry['importance']:.4f}")

Berechne globale XGBoost-Erklärung (SHAP über Trainingsset)...
Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/explanations/global_xgb_poisson_log.json

Top-5 Features (XGBoost):
  # 1  hr            0.8326
  # 2  yr            0.2004
  # 3  temp          0.1971
  # 4  weekday       0.1728
  # 5  mnth          0.1183


In [4]:
print("Berechne globale EBM-Erklärung...")
global_ebm = build_global(ebm, "ebm", X_train, metrics["ebm"])

path = save_explanation(global_ebm, f"global_ebm_{LOSS_KEY}.json")
print(f"Gespeichert: {path}")

print("\nTop-5 Features (EBM):")
for entry in global_ebm["global_importance"][:5]:
    print(f"  #{entry['rank']:2d}  {entry['feature']:12s}  {entry['importance']:.4f}")

Berechne globale EBM-Erklärung...
Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_XAI_Stahl_SoSe26/explanations/global_ebm_poisson_log.json

Top-5 Features (EBM):
  # 1  hr            0.8888
  # 2  temp          0.2526
  # 3  yr            0.2166
  # 4  mnth          0.0857
  # 5  hum           0.0826


## 3 · Lokale Erklärungen für die 5 Testinstanzen

In [5]:
for iid in INSTANCE_IDS:
    local = build_local(xgb, "xgb", X_test, y_test, iid)
    path = save_explanation(local, f"local_xgb_{LOSS_KEY}_inst{iid}.json")
    top = local["contributions"][0]
    print(f"  inst={iid:4d}  pred={local['prediction']:6.1f}  y_true={local['y_true']:6.0f}  "
          f"top_feature={top['feature']} ({top['contribution']:+.3f})  -> {path.name}")

  inst= 224  pred= 112.0  y_true=    96  top_feature=temp (-0.549)  -> local_xgb_poisson_log_inst224.json
  inst= 580  pred=  64.6  y_true=    63  top_feature=hr (-0.365)  -> local_xgb_poisson_log_inst580.json
  inst=1041  pred= 385.4  y_true=   387  top_feature=hr (+0.746)  -> local_xgb_poisson_log_inst1041.json
  inst=1481  pred= 228.6  y_true=   277  top_feature=hr (+0.378)  -> local_xgb_poisson_log_inst1481.json
  inst=1677  pred= 254.3  y_true=   286  top_feature=hr (+0.252)  -> local_xgb_poisson_log_inst1677.json
  inst=2058  pred= 194.1  y_true=   243  top_feature=yr (-0.195)  -> local_xgb_poisson_log_inst2058.json
  inst=2510  pred= 326.0  y_true=   372  top_feature=hr (+0.686)  -> local_xgb_poisson_log_inst2510.json
  inst=3543  pred= 309.9  y_true=   286  top_feature=yr (+0.186)  -> local_xgb_poisson_log_inst3543.json
  inst=3847  pred= 585.8  y_true=   531  top_feature=hr (+0.413)  -> local_xgb_poisson_log_inst3847.json
  inst=4454  pred= 297.3  y_true=   354  top_feature=hr

In [6]:
for iid in INSTANCE_IDS:
    local = build_local(ebm, "ebm", X_test, y_test, iid)
    path = save_explanation(local, f"local_ebm_{LOSS_KEY}_inst{iid}.json")
    top = local["contributions"][0]
    print(f"  inst={iid:4d}  pred={local['prediction']:6.1f}  y_true={local['y_true']:6.0f}  "
          f"top_feature={top['feature']} ({top['contribution']:+.3f})  -> {path.name}")

  inst= 224  pred= 117.0  y_true=    96  top_feature=hr (+0.852)  -> local_ebm_poisson_log_inst224.json
  inst= 580  pred=  73.1  y_true=    63  top_feature=yr (-0.226)  -> local_ebm_poisson_log_inst580.json
  inst=1041  pred= 389.7  y_true=   387  top_feature=hr (+1.109)  -> local_ebm_poisson_log_inst1041.json
  inst=1481  pred= 241.8  y_true=   277  top_feature=hr (+0.819)  -> local_ebm_poisson_log_inst1481.json
  inst=1677  pred= 260.2  y_true=   286  top_feature=hr (+0.555)  -> local_ebm_poisson_log_inst1677.json
  inst=2058  pred= 176.9  y_true=   243  top_feature=hr (+0.555)  -> local_ebm_poisson_log_inst2058.json
  inst=2510  pred= 311.4  y_true=   372  top_feature=hr (+1.148)  -> local_ebm_poisson_log_inst2510.json
  inst=3543  pred= 296.0  y_true=   286  top_feature=hr (+0.555)  -> local_ebm_poisson_log_inst3543.json
  inst=3847  pred= 593.4  y_true=   531  top_feature=hr (+0.661)  -> local_ebm_poisson_log_inst3847.json
  inst=4454  pred= 331.8  y_true=   354  top_feature=hr (

## 4 · Überblick gespeicherter Dateien

In [7]:
files = sorted(EXPLANATIONS_DIR.glob(f"*_{LOSS_KEY}*.json"))
print(f"Gespeicherte Erklärungsdateien ({len(files)}):")
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45s}  {size_kb:.1f} KB")

Gespeicherte Erklärungsdateien (22):
  global_ebm_poisson_log.json                    2.8 KB
  global_xgb_poisson_log.json                    2.8 KB
  local_ebm_poisson_log_inst1041.json            1.2 KB
  local_ebm_poisson_log_inst1481.json            1.2 KB
  local_ebm_poisson_log_inst1677.json            1.2 KB
  local_ebm_poisson_log_inst2058.json            1.2 KB
  local_ebm_poisson_log_inst224.json             1.1 KB
  local_ebm_poisson_log_inst2510.json            1.2 KB
  local_ebm_poisson_log_inst3543.json            1.2 KB
  local_ebm_poisson_log_inst3847.json            1.2 KB
  local_ebm_poisson_log_inst4454.json            1.2 KB
  local_ebm_poisson_log_inst580.json             1.2 KB
  local_xgb_poisson_log_inst1041.json            1.1 KB
  local_xgb_poisson_log_inst1481.json            1.2 KB
  local_xgb_poisson_log_inst1677.json            1.2 KB
  local_xgb_poisson_log_inst2058.json            1.2 KB
  local_xgb_poisson_log_inst224.json             1.1 KB
  local_xgb

## 5 · Sanity-Check: XGBoost vs. EBM — Top-Features im Vergleich

In [8]:
imp_xgb = {e["feature"]: e["rank"] for e in global_xgb["global_importance"]}
imp_ebm = {e["feature"]: e["rank"] for e in global_ebm["global_importance"]}

all_features = sorted(set(imp_xgb) | set(imp_ebm))
rows = [{"Feature": f, "Rank XGB": imp_xgb.get(f, "-"), "Rank EBM": imp_ebm.get(f, "-")} for f in all_features]
df = pd.DataFrame(rows).sort_values("Rank XGB")
print(df.to_string(index=False))

   Feature  Rank XGB  Rank EBM
        hr         1         1
        yr         2         3
      temp         3         2
   weekday         4         7
      mnth         5         4
       hum         6         5
weathersit         7         6
 windspeed         8         8
   holiday         9         9
